# 수업 당시 코드 참고본

과제 이후 포트폴리오 정리 과정에서 출력과 개인 PC 경로를 제거한 참고본입니다. 원본 계산의 문제도 기록으로 남아 있습니다. 재실행은 `src/fomo_eda.py`를 사용하세요. 수정 사항은 `docs/review_status.md`에 설명했습니다.


In [ ]:
import pandas as pd
import numpy as np
import glob
import matplotlib.pyplot as plt
from scipy.signal import correlate
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans


In [ ]:
base_path = r"<LOCAL_PATH>"


In [ ]:
stocks = [
    "삼성전자", "LG에너지솔루션", "에코프로", "포스코퓨처엠", "카카오",
    "네이버", "셀트리온", "현대차", "한화오션", "펄어비스"
]
naver_path = base_path + r"\네이버 데이터 랩.xlsx"

naver = pd.read_excel(naver_path)
naver = naver.rename(columns={"일자": "date"})
naver["date"] = pd.to_datetime(naver["date"])


In [ ]:
bigk_path = base_path + r"\빅카이즈 키워드.xlsx"

bigk = pd.read_excel(bigk_path)
bigk = bigk.rename(columns={"일자": "date"})
bigk["date"] = pd.to_datetime(bigk["date"])


In [ ]:
# 검색량 매핑
search_map = {s: c for s in stocks for c in naver.columns if s in c}

# 뉴스 건수 매핑
news_map = {s: c for s in stocks for c in bigk.columns if s in c}

print("검색량 매핑:", search_map)
print("뉴스 매핑:", news_map)


In [ ]:
price_files = glob.glob(base_path + r"\*.xlsx")
price_dict = {}

for s in stocks:
    for f in price_files:
        if s in f and "주식" in f:
            df = pd.read_excel(f)
            df = df.rename(columns={"일자": "date"})
            df["date"] = pd.to_datetime(df["date"])
            price_dict[s] = df
            break


In [ ]:
price_files = glob.glob(base_path + r"\*.xlsx")
price_dict = {}

for s in stocks:
    matched = [f for f in price_files if s in f]
    if len(matched) == 0:
        print("❌ 주가 파일 없음:", s)
        continue

    df = pd.read_excel(matched[0])
    df = df.rename(columns={"일자": "date"})
    df["date"] = pd.to_datetime(df["date"])
    price_dict[s] = df


In [ ]:
result_dict = {}

for s in stocks:
    price = price_dict[s]
    search_col = search_map[s]
    news_col = news_map[s]

    temp = pd.merge(price, naver[["date", search_col]], on="date", how="left")
    temp = pd.merge(temp, bigk[["date", news_col]], on="date", how="left")

    temp = temp.sort_values("date").fillna(method="ffill")

    result_dict[s] = temp

print("🎉 모든 종목 병합 완료! result_dict 생성")


In [ ]:
def flow_plot(df, stock):
    search_col = search_map[stock]
    news_col = news_map[stock]

    plt.figure(figsize=(14,8))
    plt.plot(df["date"], df[news_col], label="뉴스 건수")
    plt.plot(df["date"], df[search_col], label="검색량")
    plt.plot(df["date"], df["개인(순매수)"], label="개인 순매수")
    plt.plot(df["date"], df["종가"], label="주가")

    plt.title(f"{stock} - 뉴스→검색→순매수→주가 흐름")
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib

# Windows 기본 한글 폰트: Malgun Gothic
font_path = "C:/Windows/Fonts/malgun.ttf"
font_name = fm.FontProperties(fname=font_path).get_name()

plt.rc('font', family=font_name)
matplotlib.rcParams['axes.unicode_minus'] = False


In [ ]:
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

def flow_plot(df, stock):
    search_col = f"{stock}_x"
    news_col = f"{stock}_y"

    # 필요한 4개 컬럼만 선택
    temp = df[["date", news_col, search_col, "개인(순매수)", "종가"]].copy()

    # 정규화
    scaler = MinMaxScaler()
    temp_scaled = scaler.fit_transform(temp[[news_col, search_col, "개인(순매수)", "종가"]])
    
    temp_scaled = pd.DataFrame(temp_scaled, columns=[news_col, search_col, "개인(순매수)_scaled", "종가_scaled"])
    temp_scaled["date"] = temp["date"].values

    # 시각화
    plt.figure(figsize=(14,8))
    plt.plot(temp_scaled["date"], temp_scaled[news_col], label="뉴스 건수")
    plt.plot(temp_scaled["date"], temp_scaled[search_col], label="검색량")
    plt.plot(temp_scaled["date"], temp_scaled["개인(순매수)_scaled"], label="개인 순매수")
    plt.plot(temp_scaled["date"], temp_scaled["종가_scaled"], label="주가")

    plt.title(f"{stock} 시계열 흐름 (정규화 적용)")
    plt.legend()
    plt.grid()
    plt.show()


In [ ]:
flow_plot(result_dict["삼성전자"], "삼성전자")


In [ ]:
print(result_dict["삼성전자"][["삼성전자_x", "삼성전자_y"]].head(20))


In [ ]:
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import pandas as pd

def flow_plot(df, stock):
    # 실제 병합된 컬럼명
    search_col = f"{stock}_x"      # 검색량
    news_col = f"{stock}_y"        # 뉴스 건수

    # 원하는 4개 컬럼 전체 선택
    temp = df[["date", news_col, search_col, "개인(순매수)", "종가"]].copy()

    # 정규화 대상
    cols_to_scale = [news_col, search_col, "개인(순매수)", "종가"]

    # 정규화 수행
    scaler = MinMaxScaler()
    temp_scaled = scaler.fit_transform(temp[cols_to_scale])
    temp_scaled = pd.DataFrame(temp_scaled, columns=[f"{c}_scaled" for c in cols_to_scale])

    # 날짜 추가
    temp_scaled["date"] = temp["date"].values

    # 시각화
    plt.figure(figsize=(14,8))
    plt.plot(temp_scaled["date"], temp_scaled[f"{news_col}_scaled"], label="뉴스 건수")
    plt.plot(temp_scaled["date"], temp_scaled[f"{search_col}_scaled"], label="검색량")
    plt.plot(temp_scaled["date"], temp_scaled["개인(순매수)_scaled"], label="개인 순매수")
    plt.plot(temp_scaled["date"], temp_scaled["종가_scaled"], label="주가")

    plt.title(f"{stock} 시계열 흐름 (정규화 적용 완료)")
    plt.legend()
    plt.grid()
    plt.show()


In [ ]:
flow_plot(result_dict["삼성전자"], "삼성전자")


In [ ]:
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

def flow_plot(df, stock):

    # 실제 검색량/뉴스 컬럼
    search_col = f"{stock}_x"
    news_col = f"{stock}_y"

    # 종가 자동 탐색
    price_col = None
    for c in df.columns:
        if c == "종가" or "Close" in c or "close" in c:
            price_col = c
            break

    # 개인 순매수 자동 탐색
    order_col = None
    for c in df.columns:
        if "개인" in c and ("순" in c or "수" in c):
            order_col = c
            break

    if price_col is None or order_col is None:
        print(f"⚠ {stock}: 필수 컬럼 누락 → price_col={price_col}, order_col={order_col}")
        return
    
    # 정규화 준비
    temp = df[["date", news_col, search_col, order_col, price_col]].copy()
    
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(temp[[news_col, search_col, order_col, price_col]])
    scaled = pd.DataFrame(scaled, columns=[
        f"{news_col}_scaled", f"{search_col}_scaled", "개인순매수_scaled", "종가_scaled"
    ])
    scaled["date"] = temp["date"]

    # 그래프
    plt.figure(figsize=(14,8))
    plt.plot(scaled["date"], scaled[f"{news_col}_scaled"], label="뉴스 건수")
    plt.plot(scaled["date"], scaled[f"{search_col}_scaled"], label="검색량")
    plt.plot(scaled["date"], scaled["개인순매수_scaled"], label="개인 순매수")
    plt.plot(scaled["date"], scaled["종가_scaled"], label="주가")

    plt.title(f"{stock} 시계열 흐름 (정규화)")
    plt.legend()
    plt.grid()
    plt.show()


In [ ]:
for stock in stocks:
    print(f"\n===== {stock} =====")
    flow_plot(result_dict[stock], stock)


In [ ]:
stocks = [
    "삼성전자", "LG에너지솔루션", "에코프로", 
    "포스코퓨처엠", "카카오", "셀트리온",
    "현대차", "한화오션", "펄어비스"
]


In [ ]:
if "네이버" in result_dict:
    result_dict.pop("네이버")


In [ ]:
import numpy as np
import pandas as pd

cluster_data = []

for stock in stocks:
    df = result_dict[stock].copy()

    news_col = f"{stock}_y"
    search_col = f"{stock}_x"

    # 1) 순매수 컬럼 탐색
    order_col = None
    for c in df.columns:
        if "개인" in c and ("순" in c or "수" in c):
            order_col = c
            break

    # 1-2) 순매수 컬럼이 없으면 직접 계산
    if order_col is None:
        buy_col = None
        sell_col = None
        for c in df.columns:
            if "개인" in c and "매수" in c:
                buy_col = c
            if "개인" in c and "매도" in c:
                sell_col = c
        if buy_col and sell_col:
            df["개인_순매수_tmp"] = df[buy_col] - df[sell_col]
            order_col = "개인_순매수_tmp"
        else:
            print(f"⚠ {stock}: 순매수 데이터 없음 → 스킵")
            continue

    # 2) feature 계산
    news_std = df[news_col].std()
    search_std = df[search_col].std()
    order_std = df[order_col].std()

    # 3) lag correlation (검색량 → 순매수)
    a = df[search_col].fillna(0).values
    b = df[order_col].fillna(0).values
    corr = np.correlate(a - a.mean(), b - b.mean(), mode="full")
    lag_corr = corr.max()

    # 저장
    cluster_data.append([news_std, search_std, order_std, lag_corr])

# 결과 DataFrame
cluster_df = pd.DataFrame(cluster_data,
    columns=["뉴스_STD", "검색량_STD", "순매수_STD", "LagCorr"],
    index=stocks
)

cluster_df


In [ ]:
from sklearn.cluster import KMeans

k = 3
model = KMeans(n_clusters=k, random_state=42)
cluster_df["cluster"] = model.fit_predict(cluster_df)

print(cluster_df)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
plt.scatter(cluster_df["검색량_STD"], cluster_df["순매수_STD"],
            c=cluster_df["cluster"], cmap="viridis", s=200)

for stock in stocks:
    plt.text(cluster_df.loc[stock, "검색량_STD"],
             cluster_df.loc[stock, "순매수_STD"],
             stock, fontsize=12)

plt.xlabel("검색량 변동성 STD")
plt.ylabel("개인 순매수 변동성 STD")
plt.title("FOMO 패턴 기반 종목 클러스터링 (네이버 제외)")
plt.grid()
plt.show()


In [ ]:
print(cluster_df)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,8))
sns.heatmap(cluster_df.drop(columns=["cluster"]).corr(), annot=True, cmap="coolwarm")
plt.title("FOMO Feature Correlation Heatmap")
plt.show()


In [ ]:
from statsmodels.graphics.tsaplots import plot_ccf

df = result_dict["삼성전자"]
plt.figure(figsize=(10,4))
plot_ccf(df["삼성전자_x"].fillna(0), df["개인(순매수)"].fillna(0))
plt.title("삼성전자 검색량 → 순매수 CCF")
plt.show()


In [ ]:
plt.figure(figsize=(10,5))
cluster_df[["뉴스_STD", "검색량_STD", "순매수_STD"]].boxplot()
plt.title("종목별 FOMO 변수 변동성 비교")
plt.show()


In [ ]:
import numpy as np

df = result_dict["에코프로"]
threshold = df["에코프로_x"].mean() + df["에코프로_x"].std()*2
spike = df[df["에코프로_x"] > threshold]

plt.figure(figsize=(14,6))
plt.plot(df["date"], df["에코프로_x"], label="검색량")
plt.scatter(spike["date"], spike["에코프로_x"], color='red', label="Spike")
plt.legend()
plt.title("에코프로 검색량 Spike Detection")
plt.show()


In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaled = scaler.fit_transform(cluster_df[["뉴스_STD","검색량_STD","순매수_STD","LagCorr"]])
cluster_df["FOMO_Index"] = scaled.mean(axis=1)

cluster_df["FOMO_Index"].plot(kind="bar", figsize=(10,5), title="FOMO Index by Stock")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_spike_all():
    for stock in stocks:  # 네이버 제외한 9개 종목
        df = result_dict[stock].copy()
        search_col = f"{stock}_x"

        # 스파이크 기준 계산
        threshold = df[search_col].mean() + df[search_col].std() * 2

        spike_df = df[df[search_col] > threshold]

        plt.figure(figsize=(14,6))
        plt.plot(df["date"], df[search_col], label="검색량")
        plt.scatter(spike_df["date"], spike_df[search_col], 
                    color='red', s=80, label="Spike (검색량 급등)", zorder=3)
        
        plt.axhline(threshold, color='orange', linestyle='--', label="Spike 기준")
        plt.title(f"{stock} 검색량 Spike Detection", fontsize=14)
        plt.xlabel("Date")
        plt.ylabel("검색량")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.show()

# 실행
plot_spike_all()


In [ ]:
import pandas as pd

def detect_spike(series):
    threshold = series.mean() + series.std() * 2
    return series > threshold, threshold

pairing_results = []

for stock in stocks:
    df = result_dict[stock].copy()
    
    search_col = f"{stock}_x"
    
    # 순매수 컬럼 자동 탐색
    order_col = None
    for c in df.columns:
        if "개인" in c and ("순" in c or "수" in c):
            order_col = c
            break
            
    if order_col is None:
        print(f"⚠ 순매수 컬럼 없음 → {stock}")
        continue
    
    # Spike 탐지
    search_spike, _ = detect_spike(df[search_col])
    order_spike, _ = detect_spike(df[order_col])
    
    # 날짜 리스트
    search_dates = df.loc[search_spike, "date"]
    order_dates = df.loc[order_spike, "date"]

    # 0~3일 후 반응 계산
    same_day = set(search_dates).intersection(set(order_dates))
    day1 = set(search_dates + pd.Timedelta(days=1)).intersection(set(order_dates))
    day2 = set(search_dates + pd.Timedelta(days=2)).intersection(set(order_dates))
    day3 = set(search_dates + pd.Timedelta(days=3)).intersection(set(order_dates))

    total_pairs = len(same_day) + len(day1) + len(day2) + len(day3)

    pairing_results.append({
        "종목": stock,
        "검색량 Spike 수": len(search_dates),
        "순매수 Spike 수": len(order_dates),
        "동일날 Spike": len(same_day),
        "1일 후 반응": len(day1),
        "2일 후 반응": len(day2),
        "3일 후 반응": len(day3),
        "반응 없음": len(search_dates) - total_pairs     # 남은 검색량 Spike
    })

pair_df = pd.DataFrame(pairing_results)
pair_df


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Heatmap용 DF
heatmap_df = pair_df[["동일날 Spike","1일 후 반응","2일 후 반응","3일 후 반응"]]
heatmap_df.index = pair_df["종목"]

plt.figure(figsize=(10,7))
sns.heatmap(heatmap_df, annot=True, fmt="d", cmap="YlOrRd")
plt.title("검색량 Spike → 순매수 Spike 반응 Heatmap (0~3일)")
plt.ylabel("종목")
plt.xlabel("반응 시점")
plt.show()


In [ ]:
plt.figure(figsize=(12,6))
pair_df.set_index("종목")[["동일날 Spike","1일 후 반응","2일 후 반응","3일 후 반응"]].plot(
    kind="bar", figsize=(14,6)
)
plt.title("종목별 FOMO Spike Pair Distribution")
plt.ylabel("Spike Count")
plt.xlabel("종목")
plt.grid(alpha=0.3)
plt.show()


In [ ]:
df_raw = result_dict["삼성전자"].copy()

# 필요한 변수 이름 정리
news_col = "삼성전자_y"
search_col = "삼성전자_x"
order_col = "개인(순매수)"
price_col = "종가"

df = df_raw[["date", news_col, search_col, order_col, price_col]].dropna()
df = df.set_index("date")
df.head()


In [ ]:
import numpy as np

df_log = np.log(df + 1)

df_diff = df_log.diff().dropna()
df_diff.head()


In [ ]:
from statsmodels.tsa.api import VAR

model = VAR(df_diff)
lag_selection = model.select_order(maxlags=10)
lag_selection.summary()


In [ ]:
lag = lag_selection.aic

var_model = model.fit(lag)
print(var_model.summary())


In [ ]:
print(var_model.test_causality(caused=search_col, causing=news_col).summary())
print(var_model.test_causality(caused=order_col, causing=search_col).summary())
print(var_model.test_causality(caused=price_col, causing=order_col).summary())


In [ ]:
irf = var_model.irf(10)  # 10일 동안 충격 반응 관찰
irf.plot(orth=False)


In [ ]:
fevd = var_model.fevd(10)
fevd.plot()


In [ ]:
import pandas as pd
import numpy as np

# --------------------------------------------------
# 0. 기본 설정
# --------------------------------------------------

stocks = [
    "삼성전자", "LG에너지솔루션", "에코프로",
    "포스코퓨처엠", "카카오", "셀트리온",
    "현대차", "한화오션", "펄어비스"
]

price_files = {
    "삼성전자": "<LOCAL_PATH>",
    "LG에너지솔루션": "<LOCAL_PATH>",
    "에코프로": "<LOCAL_PATH>",
    "포스코퓨처엠": "<LOCAL_PATH>",
    "카카오": "<LOCAL_PATH>",
    "셀트리온": "<LOCAL_PATH>",
    "현대차": "<LOCAL_PATH>",
    "한화오션": "<LOCAL_PATH>",
    "펄어비스": "<LOCAL_PATH>"
}


# --------------------------------------------------
# 1. 날짜 정제 함수
# --------------------------------------------------

def fix_date_column(df):
    for cand in ["일자", "날짜", "DATE", "Date"]:
        if cand in df.columns:
            df = df.rename(columns={cand: "date"})
            break

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    df = df.drop_duplicates(subset=["date"])
    return df.sort_values("date").reset_index(drop=True)


# --------------------------------------------------
# 2. 파일 로딩 + 날짜 정제
# --------------------------------------------------

price_dict = {}

for s in stocks:
    df = pd.read_excel(price_files[s])
    df = fix_date_column(df)
    price_dict[s] = df


# --------------------------------------------------
# 3. 네이버 / 빅카인즈 로딩 및 날짜 정제
# --------------------------------------------------

naver = pd.read_excel("<LOCAL_PATH>")
naver = fix_date_column(naver)

bigk = pd.read_excel("<LOCAL_PATH>")
bigk = fix_date_column(bigk)


# --------------------------------------------------
# 4. 병합 (검색량, 뉴스량 컬럼 구분)
# --------------------------------------------------

merged_dict = {}

for s in stocks:
    df_price = price_dict[s].copy()

    # 검색량 컬럼 → {s}_search
    nav = naver[["date", s]].rename(columns={s: f"{s}_search"})

    # 뉴스량 컬럼 → {s}_news
    news = bigk[["date", s]].rename(columns={s: f"{s}_news"})

    # 병합
    df_merge = (
        df_price
        .merge(nav, on="date", how="left")
        .merge(news, on="date", how="left")
    )

    merged_dict[s] = df_merge


# --------------------------------------------------
# 5. 결측치 처리
# --------------------------------------------------

clean_null_dict = {}

for s in stocks:
    df = merged_dict[s].copy()

    # 날짜 기준 정렬
    df = df.sort_values("date")

    # 결측치 처리 (전방/후방 보간 + 0 채우기)
    df = df.ffill().bfill()

    clean_null_dict[s] = df


# --------------------------------------------------
# 6. 이상치 제거 (IQR)
# --------------------------------------------------

final_dict = {}

for s in stocks:
    df = clean_null_dict[s].copy()

    search_col = f"{s}_search"
    news_col = f"{s}_news"
    order_cols = [c for c in df.columns if "순매수" in c]
    order_col = order_cols[0] if order_cols else None

    def clean(series):
        Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
        IQR = Q3 - Q1
        low, high = Q1 - 1.5*IQR, Q3 + 1.5*IQR
        return series.clip(low, high)

    # 적용
    if search_col in df:
        df[search_col] = clean(df[search_col])
    if news_col in df:
        df[news_col] = clean(df[news_col])
    if order_col in df:
        df[order_col] = clean(df[order_col])

    final_dict[s] = df


# --------------------------------------------------
# 7. 최종 전처리 결과 확인
# --------------------------------------------------

for s in stocks:
    print(f"\n📌 전처리 완료: {s}")
    display(final_dict[s].head())

print("\n🎉 전처리(날짜 정제 + 결측치 보간 + 이상치 제거 + 병합) 완료!")


In [ ]:
import pandas as pd
import numpy as np

# ---------------------------------------------------
# 1. IQR 기반 이상치 탐지 함수
# ---------------------------------------------------

def detect_outliers(series):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    low, high = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

    outliers = series[(series < low) | (series > high)]
    return outliers, low, high


# ---------------------------------------------------
# 2. 각 종목별 이상치 탐지 + 원인 분석 자동화
# ---------------------------------------------------

def generate_outlier_report(final_dict, stocks):
    report = ""

    for s in stocks:
        df = final_dict[s].copy()

        search_col = f"{s}_search"
        news_col   = f"{s}_news"

        # 존재 여부 확인
        if search_col not in df.columns or news_col not in df.columns:
            report += f"\n❗ {s}: 검색량 또는 뉴스량 컬럼이 없어 분석 불가\n"
            continue

        # 1) 검색량 이상치 탐지
        search_outliers, low_s, high_s = detect_outliers(df[search_col])

        # 2) 뉴스량 이상치 탐지
        news_outliers, low_n, high_n = detect_outliers(df[news_col])

        report += f"\n\n=============================="
        report += f"\n📌 [종목] {s}"
        report += f"\n=============================="

        # ------------------------------------------------------------
        # 검색량 이상치 분석
        # ------------------------------------------------------------
        if len(search_outliers) > 0:
            report += f"\n\n🔎 검색량 이상치 발생 ({len(search_outliers)}건)"
            for date, val in search_outliers.items():
                row = df[df["date"] == date]
                news_val = row[news_col].values[0]
                close_val = row["종가"].values[0]
                
                # 원인 추정 로직 (뉴스량 vs 검색량)
                if news_val > high_n:
                    reason = "→ 뉴스 급증이 원인 가능성이 높음"
                else:
                    reason = "→ 이슈 or 커뮤니티 확산 가능성"

                report += f"\n  - 날짜: {date.date()} | 검색량: {val:.0f} | 종가: {close_val:,} | {reason}"

        else:
            report += "\n\n🔎 검색량 이상치 없음"

        # ------------------------------------------------------------
        # 뉴스량 이상치 분석
        # ------------------------------------------------------------
        if len(news_outliers) > 0:
            report += f"\n\n📰 뉴스량 이상치 발생 ({len(news_outliers)}건)"

            for date, val in news_outliers.items():
                row = df[df["date"] == date]
                search_val = row[search_col].values[0]
                close_val = row["종가"].values[0]

                if search_val > high_s:
                    reason = "→ 검색량도 증가 → 대형 이슈 가능성 매우 높음"
                else:
                    reason = "→ 언론 주도 보도 증가 가능성"

                report += f"\n  - 날짜: {date.date()} | 뉴스량: {val:.0f} | 종가: {close_val:,} | {reason}"

        else:
            report += "\n\n📰 뉴스량 이상치 없음"

    return report


# ---------------------------------------------------
# 3. 실행 (final_dict 사용)
# ---------------------------------------------------

report_text = generate_outlier_report(final_dict, stocks)

print(report_text)


In [ ]:
def add_features(df, stock):
    s_search = f"{stock}_search"
    s_news   = f"{stock}_news"

    # --- 주가 ---
    df["ret_1d"] = df["종가"].pct_change()
    df["ret_5d"] = df["종가"].pct_change(5)
    df["ret_20d"] = df["종가"].pct_change(20)

    df["vol_5d"] = df["ret_1d"].rolling(5).std()
    df["vol_20d"] = df["ret_1d"].rolling(20).std()

    df["ma_5"] = df["종가"].rolling(5).mean()
    df["ma_20"] = df["종가"].rolling(20).mean()
    df["ma_60"] = df["종가"].rolling(60).mean()

    df["gap_5_20"] = df["ma_5"] - df["ma_20"]
    df["gap_20_60"] = df["ma_20"] - df["ma_60"]

    # --- 검색량 ---
    df["search_change"] = df[s_search].pct_change()
    df["search_diff"] = df[s_search].diff()
    df["search_ma_7"] = df[s_search].rolling(7).mean()
    df["search_ma_30"] = df[s_search].rolling(30).mean()

    # --- 뉴스량 ---
    df["news_change"] = df[s_news].pct_change()
    df["news_ma_7"] = df[s_news].rolling(7).mean()
    df["news_ma_30"] = df[s_news].rolling(30).mean()

    # --- 이상치 flag ---
    Q1_s, Q3_s = df[s_search].quantile(0.25), df[s_search].quantile(0.75)
    IQR_s = Q3_s - Q1_s
    low_s, high_s = Q1_s - 1.5 * IQR_s, Q3_s + 1.5 * IQR_s

    df["is_search_outlier"] = ((df[s_search] < low_s) | (df[s_search] > high_s)).astype(int)

    Q1_n, Q3_n = df[s_news].quantile(0.25), df[s_news].quantile(0.75)
    IQR_n = Q3_n - Q1_n
    low_n, high_n = Q1_n - 1.5 * IQR_n, Q3_n + 1.5 * IQR_n

    df["is_news_outlier"] = ((df[s_news] < low_n) | (df[s_news] > high_n)).astype(int)

    # 검색량 spike 이후 수익률
    df["ret_after_search_1d"] = df["ret_1d"].shift(-1)
    df["ret_after_search_5d"] = df["ret_5d"].shift(-5)

    return df
